# 11 - Final Prediction & Demonstration
**Metro Traffic Flow Prediction - Real-Time Inference, Scenarios & Sensitivity Analysis**

This notebook demonstrates the end-to-end inference workflow using the serialized production model, testing 4 real-world operational scenarios, sensitivity perturbation, and validation on unseen data.

---
### 1. Load Production Model & Scaler

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

models_dir = root / "models"
model_path = models_dir / "hybridensemble.pkl"
if not model_path.exists():
    model_path = models_dir / "xgboost.pkl"

scaler_path = models_dir / "preprocessor.pkl"

production_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print(f"Loaded Production Model: {type(production_model).__name__}")
print(f"Loaded Scaler with {len(scaler.feature_names_in_)} expected features.")

Loaded Production Model: StackingRegressor
Loaded Scaler with 28 expected features.


---
### 2. Standalone Inference Pipeline Function

In [2]:
def predict_metro_flow(input_data: dict) -> float:
    """
    Predict metro traffic flow given operational, temporal, and weather inputs.
    """
    df_in = pd.DataFrame([input_data])
    feature_cols = scaler.feature_names_in_
    
    for col in feature_cols:
        if col not in df_in.columns:
            df_in[col] = 0.0
            
    df_ordered = df_in[feature_cols]
    df_scaled = pd.DataFrame(scaler.transform(df_ordered), columns=feature_cols)
    prediction = production_model.predict(df_scaled)[0]
    return round(float(prediction), 2)

---
### 3. Demonstration Across 4 Real-World Scenarios
1. **Scenario A - Morning Peak Rush Hour:** High volume, peak flag active, weekday morning.
2. **Scenario B - Late-Night Quiet Service:** Low volume, midnight service, off-peak.
3. **Scenario C - Adverse Weather & Signal Delay:** Heavy monsoon rain with signal delay.
4. **Scenario D - Weekend Special Event:** Major stadium event with elevated occupancy.

In [3]:
scenarios = [
    {
        "Scenario": "A: Morning Peak Rush Hour",
        "Station_ID": 5, "Line_ID": 2, "Train_ID": 104, "Hour": 8.0, "Day_of_Week": 1,
        "Weekend_Flag": 0, "Holiday_Flag": 0, "Special_Event_Flag": 0, "Peak_Hour_Flag": 1,
        "Passenger_Count": 420, "Boardings": 280, "Alightings": 140,
        "Ticket_Entries": 310, "Ticket_Exits": 160, "Train_Frequency_per_Hour": 12.0,
        "Average_Dwell_Time_sec": 45.0, "Average_Headway_min": 5.0, "Average_Speed_kmph": 52.0,
        "Occupancy_Rate_pct": 88.0, "Platform_Density": 4.2, "Temperature_C": 24.0,
        "Humidity_pct": 65.0, "Rainfall_mm": 0.0, "Visibility_km": 10.0, "Signal_Delay_sec": 0.0,
        "Incident_Flag": 0, "Maintenance_Flag": 0, "Energy_Consumption_kWh": 1650.0
    },
    {
        "Scenario": "B: Late-Night Quiet Service",
        "Station_ID": 12, "Line_ID": 1, "Train_ID": 208, "Hour": 23.5, "Day_of_Week": 2,
        "Weekend_Flag": 0, "Holiday_Flag": 0, "Special_Event_Flag": 0, "Peak_Hour_Flag": 0,
        "Passenger_Count": 45, "Boardings": 25, "Alightings": 30,
        "Ticket_Entries": 30, "Ticket_Exits": 35, "Train_Frequency_per_Hour": 4.0,
        "Average_Dwell_Time_sec": 20.0, "Average_Headway_min": 15.0, "Average_Speed_kmph": 65.0,
        "Occupancy_Rate_pct": 15.0, "Platform_Density": 0.8, "Temperature_C": 18.0,
        "Humidity_pct": 75.0, "Rainfall_mm": 0.0, "Visibility_km": 10.0, "Signal_Delay_sec": 0.0,
        "Incident_Flag": 0, "Maintenance_Flag": 0, "Energy_Consumption_kWh": 950.0
    },
    {
        "Scenario": "C: Adverse Weather & Signal Delay",
        "Station_ID": 3, "Line_ID": 2, "Train_ID": 115, "Hour": 8.5, "Day_of_Week": 3,
        "Weekend_Flag": 0, "Holiday_Flag": 0, "Special_Event_Flag": 0, "Peak_Hour_Flag": 1,
        "Passenger_Count": 380, "Boardings": 240, "Alightings": 150,
        "Ticket_Entries": 260, "Ticket_Exits": 180, "Train_Frequency_per_Hour": 8.0,
        "Average_Dwell_Time_sec": 65.0, "Average_Headway_min": 7.5, "Average_Speed_kmph": 38.0,
        "Occupancy_Rate_pct": 92.0, "Platform_Density": 5.1, "Temperature_C": 21.0,
        "Humidity_pct": 95.0, "Rainfall_mm": 35.0, "Visibility_km": 2.5, "Signal_Delay_sec": 75.0,
        "Incident_Flag": 0, "Maintenance_Flag": 0, "Energy_Consumption_kWh": 1800.0
    },
    {
        "Scenario": "D: Weekend Special Event",
        "Station_ID": 8, "Line_ID": 3, "Train_ID": 305, "Hour": 18.0, "Day_of_Week": 5,
        "Weekend_Flag": 1, "Holiday_Flag": 0, "Special_Event_Flag": 1, "Peak_Hour_Flag": 1,
        "Passenger_Count": 460, "Boardings": 310, "Alightings": 220,
        "Ticket_Entries": 330, "Ticket_Exits": 250, "Train_Frequency_per_Hour": 10.0,
        "Average_Dwell_Time_sec": 50.0, "Average_Headway_min": 6.0, "Average_Speed_kmph": 48.0,
        "Occupancy_Rate_pct": 85.0, "Platform_Density": 4.6, "Temperature_C": 26.0,
        "Humidity_pct": 55.0, "Rainfall_mm": 0.0, "Visibility_km": 10.0, "Signal_Delay_sec": 10.0,
        "Incident_Flag": 0, "Maintenance_Flag": 0, "Energy_Consumption_kWh": 1550.0
    }
]

demo_results = []
for sc in scenarios:
    pred_flow = predict_metro_flow(sc)
    demo_results.append({
        "Scenario": sc["Scenario"],
        "Hour": sc["Hour"],
        "Passenger_Count": sc["Passenger_Count"],
        "Occupancy_Rate_pct": sc["Occupancy_Rate_pct"],
        "Predicted_Traffic_Flow": pred_flow
    })

demo_df = pd.DataFrame(demo_results)
display(demo_df)

,Scenario,Hour,Passenger_Count,Occupancy_Rate_pct,Predicted_Traffic_Flow
0,A: Morning Peak Rush Hour,8.0,420,88.0,452.12
1,B: Late-Night Quiet Service,23.5,45,15.0,125.11
2,C: Adverse Weather & Signal Delay,8.5,380,92.0,404.14
3,D: Weekend Special Event,18.0,460,85.0,438.11


---
### 4. Operational Sensitivity Perturbation Test
*(Merged from sensitivity analysis)*
We test marginal traffic flow change when operational conditions are perturbed on a baseline observation:
- Base Condition
- Perturbation: `Passenger_Count + 25`, `Boardings + 15`, `Peak_Hour_Flag = 1`

In [4]:
raw_df = pd.read_csv(root / "data" / "raw" / "metro_traffic_dataset_10000.csv")
base_row = raw_df.iloc[0].to_dict()
base_pred = predict_metro_flow(base_row)

# Perturbed condition
perturbed_row = base_row.copy()
perturbed_row["Passenger_Count"] = perturbed_row["Passenger_Count"] + 25
perturbed_row["Boardings"] = perturbed_row["Boardings"] + 15
perturbed_row["Peak_Hour_Flag"] = 1
perturbed_pred = predict_metro_flow(perturbed_row)

sens_df = pd.DataFrame([
    {"State": "Base Observation", "Passenger_Count": base_row["Passenger_Count"], "Boardings": base_row["Boardings"], "Peak_Hour_Flag": base_row["Peak_Hour_Flag"], "Predicted_Flow": base_pred},
    {"State": "Perturbed (+25 Pass, +15 Board, Peak=1)", "Passenger_Count": perturbed_row["Passenger_Count"], "Boardings": perturbed_row["Boardings"], "Peak_Hour_Flag": perturbed_row["Peak_Hour_Flag"], "Predicted_Flow": perturbed_pred}
])
sens_df["Marginal_Delta"] = [0.0, round(perturbed_pred - base_pred, 2)]
display(sens_df)

,State,Passenger_Count,Boardings,Peak_Hour_Flag,Predicted_Flow,Marginal_Delta
0,Base Observation,197,161,0,231.26,0.00
1,"Perturbed (+25 Pass, +15 Board, Peak=1)",222,176,1,267.39,36.13


---
### 5. Validation on Unseen Test Dataset Samples

In [5]:
sample_unseen = raw_df.sample(5, random_state=123)

val_results = []
for idx, row in sample_unseen.iterrows():
    row_dict = row.to_dict()
    actual = row_dict["Traffic_Flow_Target"]
    predicted = predict_metro_flow(row_dict)
    error = abs(actual - predicted)
    val_results.append({
        "Sample_Index": idx,
        "Hour": row_dict["Hour"],
        "Passenger_Count": row_dict["Passenger_Count"],
        "Actual_Flow": round(actual, 2),
        "Predicted_Flow": predicted,
        "Absolute_Error": round(error, 2),
        "Accuracy_pct": round((1 - error / actual) * 100, 2)
    })

val_df = pd.DataFrame(val_results)
display(val_df)

,Sample_Index,Hour,Passenger_Count,Actual_Flow,Predicted_Flow,Absolute_Error,Accuracy_pct
0,2656,21.00,142,193.67,202.82,9.15,95.28
1,445,20.25,168,194.18,194.88,0.70,99.64
2,9505,5.25,215,197.00,231.17,34.17,82.65
3,332,16.00,148,179.66,179.64,0.02,99.99
4,4168,15.00,215,246.66,235.65,11.01,95.54


---
### Integration Note:
The final prediction pipeline seamlessly powers the Streamlit interactive dashboard (`app/` and `08_Application/app.py`), enabling live operator querying and transit simulation.